# 03 — Entrenamiento clasificador PANTS

Entrena el clasificador multi-task ViT sobre las imágenes de **pants**.

**Pre-requisito:** ejecutar `02_prepare_data_colab.ipynb` antes para descargar
las imágenes y generar los splits.

**Heads entrenadas:**
- `color_family`, `pattern`, `fit_silhouette`, `fabric_content`, `dressing_syle`
- `waist_rise` (exclusivo de pants)


## 1. Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Autenticación exitosa.')

BUCKET     = 'gs://jbj-vision'
REPO_DIR   = '/content/repo'
CACHE_ROOT = '/content/cache'

# Sincronizar repo desde GCS
!gsutil -m rsync -r -x '\.git/.*|__pycache__/.*' {BUCKET}/code/ {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gcsfs==2023.10.0 fsspec google-cloud-storage
!pip install -q -r requirements.txt


## 2. Cargar config con paths de Drive

In [ ]:
import sys
sys.path.insert(0, '/content/repo')

from src.data.colab import setup_gcp
config = setup_gcp('config/pipeline_config.yaml', bucket='gs://jbj-vision', local_cache_root='/content/cache')
print('Backbone:', config['pants']['backbone'])
print('Splits en:', config['paths']['splits'])
print('Modelos en:', config['paths']['models'])

# Override hyperparams
config['pants']['training']['epochs'] = 15
config['pants']['training']['batch_size'] = 32
config['pants']['training']['use_class_weights'] = True
print('Heads:', list(config['pants']['heads'].keys()))


## 2.5. Restaurar cache de imágenes desde GCS

In [ ]:
import subprocess, os

def pull_tarball(ds: str) -> bool:
    gcs_path = f"{BUCKET}/data/processed/images_{ds}.tar.gz"
    local_tar = f"/tmp/images_{ds}.tar.gz"
    r = subprocess.run(['gsutil', 'ls', gcs_path], capture_output=True)
    if r.returncode == 0:
        print(f'Pulling {gcs_path} ...')
        os.makedirs(config['paths'][f'images_{ds}'], exist_ok=True)
        !gsutil -m cp {gcs_path} {local_tar}
        !tar xzf {local_tar} -C {config['paths'][f'images_{ds}']} --strip-components=1
        print(f'✓ Cache {ds} restaurado.')
        return True
    print(f'Sin tarball para {ds} — ejecutá 02_prepare_data_colab.ipynb primero.')
    return False

pull_tarball('pants')
pull_tarball('tops')


## 3. Entrenar

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

from src.classification.train import train_from_csv

result = train_from_csv(
    config=config,
    dataset_key='pants',
    train_csv=f"{config['paths']['splits']}/pants_1_train.csv",
    val_csv=f"{config['paths']['splits']}/pants_1_val.csv",
    cache_dir=config['paths']['images_pants'],
    history_path=f"{config['paths']['outputs']}/history_pants.json",
)
print(f"\n✓ Best F1-macro: {result['best_metric']:.3f}")
print(f"✓ Checkpoint: {result['checkpoint']}")


## 4. Visualizar historia de entrenamiento

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(result['history'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['epoch'], history['train_loss'], 'o-', label='train loss')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss'); axes[0].set_title('Train loss')
axes[0].grid(True)

axes[1].plot(history['epoch'], history['val_acc_mean'], 'o-', label='val acc')
axes[1].plot(history['epoch'], history['val_f1_mean'], 'o-', label='val F1-macro')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Metrica')
axes[1].set_title('Validation'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()


## 5. Métricas finales por head

In [ ]:
last = result['history'][-1]['per_head']
rows = [{
    'head': name,
    'accuracy': round(m['accuracy'], 3),
    'f1_macro': round(m['f1_macro'], 3),
    'n_val': m['n'],
} for name, m in last.items()]
import pandas as pd
display(pd.DataFrame(rows))
